# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Secret successfully loaded!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    'dim_clients':f"read_parquet('{rel}/dim_clients.parquet')",
    'dim_content':f"read_parquet('{rel}/dim_content.parquet')",
    'fact_content_daily_performance':f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    'fact_content_query_90d':f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}


Secret successfully loaded!


In [41]:
df = con.sql(f"""
WITH content_id_hv_criteria AS (
    SELECT
        content_hash_id
    FROM {TABLES['fact_content_daily_performance']}
    WHERE
        gsc_data_available IS TRUE
        AND ga4_data_available IS NOT NULL
    GROUP BY content_hash_id
    HAVING
        COUNT(DISTINCT report_date) >= 120
        AND
        DATE_DIFF(
            'day',
            MIN(report_date),
            MAX(report_date)
        ) + 1 = COUNT(DISTINCT report_date)
)

SELECT fact.*
FROM {TABLES['fact_content_daily_performance']} AS fact
INNER JOIN content_id_hv_criteria AS criteria
    ON fact.content_hash_id = criteria.content_hash_id
WHERE
    fact.gsc_data_available IS TRUE
    AND fact.ga4_data_available IS NOT NULL;
""").df()

df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-02-18,client_9958f0a7ae1df715,content_500ffc91a8fd16ae,True,True,True,False,15,0,359,...,0,0,0,0,0,0,0,0,0,2025-02
1,2025-02-18,client_9958f0a7ae1df715,content_6ac58dc72192030e,True,True,True,False,18,0,698,...,0,0,0,0,0,0,0,0,0,2025-02
2,2025-02-18,client_9958f0a7ae1df715,content_d02be57d816cf3d7,True,True,True,False,83,0,411,...,0,0,0,0,0,0,0,0,0,2025-02
3,2025-02-18,client_9958f0a7ae1df715,content_04b93a60b87a2376,True,True,True,False,32,0,144,...,0,0,0,0,0,0,0,0,0,2025-02
4,2025-02-18,client_9958f0a7ae1df715,content_dad71ac028d6a94b,True,True,True,False,1,0,0,...,0,0,0,0,0,0,0,0,0,2025-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2562564,2026-06-29,client_b10cb2997d0c7c86,content_bc8403848ecc739a,True,True,True,False,7,0,153,...,0,0,0,0,0,0,0,0,0,2026-06
2562565,2026-06-29,client_b10cb2997d0c7c86,content_f78c8f862b9bb89d,True,True,True,False,7,0,42,...,0,0,0,0,0,0,0,0,0,2026-06
2562566,2026-06-29,client_b10cb2997d0c7c86,content_b2d8ec2e5b32c6af,True,True,True,False,7,0,208,...,0,0,0,0,0,0,0,0,0,2026-06
2562567,2026-06-29,client_b10cb2997d0c7c86,content_82e319170551ad21,True,True,True,False,28,0,198,...,0,0,0,0,0,0,0,0,0,2026-06


In [66]:
result = df.groupby('content_hash_id').agg(star_date = ('report_date', "min"),end_date = ('report_date', "max")).sort_values(['star_date','end_date'], ascending=[False,True])
result

,star_date,end_date
content_hash_id,,
content_0060ca245492494f,2026-03-03,2026-06-30
content_00823406ee06d1e7,2026-03-03,2026-06-30
content_0084ee8efa2fcd23,2026-03-03,2026-06-30
content_00e0da358d5df2e7,2026-03-03,2026-06-30
content_00efcc9c1c9bba3d,2026-03-03,2026-06-30
...,...,...
content_9987be00b8c14a02,2025-01-27,2026-06-30
content_cf0500bbb61cc8b9,2025-01-27,2026-06-30
content_d02be57d816cf3d7,2025-01-27,2026-06-30


In [1]:
incomplete_content_month = result[result['star_date'].dt.day > 15]
data_complete_month = df.merge(incomplete_content_month, on='content_hash_id',how='outer')

NameError: name 'result' is not defined

In [85]:
data_complete_month_1 = df.merge(incomplete_content_month, on='content_hash_id',how='outer')

In [88]:
data_complete_month

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,star_date,end_date
0,2025-02-18,client_9958f0a7ae1df715,content_d02be57d816cf3d7,True,True,True,False,83,0,411,...,0,0,0,0,0,0,0,2025-02,2025-01-27,2026-06-30
1,2025-02-18,client_9958f0a7ae1df715,content_dad71ac028d6a94b,True,True,True,False,1,0,0,...,0,0,0,0,0,0,0,2025-02,2025-02-18,2026-06-30
2,2025-02-18,client_9958f0a7ae1df715,content_0217be03126aa7a5,True,True,True,False,45,0,453,...,0,0,0,0,0,0,0,2025-02,2025-01-27,2026-06-30
3,2025-02-18,client_9958f0a7ae1df715,content_84f5a9ecfefa108e,True,True,True,False,115,1,234,...,0,0,0,0,0,0,0,2025-02,2025-01-27,2026-06-30
4,2025-02-18,client_9958f0a7ae1df715,content_3bbd3bd840cb056c,True,True,True,False,18,0,202,...,0,0,0,0,0,0,0,2025-02,2025-02-18,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1373799,2026-06-29,client_b10cb2997d0c7c86,content_4b62c98a956400b0,True,True,True,False,15,0,463,...,0,0,0,0,0,0,0,2026-06,2026-02-22,2026-06-30
1373800,2026-06-29,client_b10cb2997d0c7c86,content_3c50a42ee52d4c20,True,True,True,True,16,0,141,...,0,0,0,0,0,0,1,2026-06,2026-02-18,2026-06-30
1373801,2026-06-29,client_b10cb2997d0c7c86,content_bc8403848ecc739a,True,True,True,False,7,0,153,...,0,0,0,0,0,0,0,2026-06,2026-02-27,2026-06-30
1373802,2026-06-29,client_b10cb2997d0c7c86,content_b2d8ec2e5b32c6af,True,True,True,False,7,0,208,...,0,0,0,0,0,0,0,2026-06,2026-02-27,2026-06-30


In [83]:
incomplete_content_month

,star_date,end_date
content_hash_id,,
content_0060ca245492494f,2026-03-03,2026-06-30
content_00823406ee06d1e7,2026-03-03,2026-06-30
content_0084ee8efa2fcd23,2026-03-03,2026-06-30
content_00e0da358d5df2e7,2026-03-03,2026-06-30
content_00efcc9c1c9bba3d,2026-03-03,2026-06-30
...,...,...
content_f70e22b3dcc80ca1,2025-02-04,2026-06-30
content_00d12fab87754f25,2025-02-03,2026-06-30
content_8bea9167e438fad6,2025-02-03,2026-06-30


In [82]:
data_complete_month

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,star_date,end_date
0,2025-02-18,client_9958f0a7ae1df715,content_500ffc91a8fd16ae,True,True,True,False,15,0,359,...,0,0,0,0,0,0,0,2025-02,2025-02-04,2026-06-30
1,2025-02-18,client_9958f0a7ae1df715,content_6ac58dc72192030e,True,True,True,False,18,0,698,...,0,0,0,0,0,0,0,2025-02,2025-02-06,2026-06-30
2,2025-02-18,client_9958f0a7ae1df715,content_04b93a60b87a2376,True,True,True,False,32,0,144,...,0,0,0,0,0,0,0,2025-02,2025-02-07,2026-06-30
3,2025-02-18,client_9958f0a7ae1df715,content_92fd77e367253da2,True,True,True,False,45,2,202,...,0,0,0,0,0,0,0,2025-02,2025-02-07,2026-06-30
4,2025-02-18,client_9958f0a7ae1df715,content_7d16dcdb89ab9219,True,True,True,False,27,0,137,...,0,0,0,0,0,0,0,2025-02,2025-02-11,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1152519,2026-06-29,client_b10cb2997d0c7c86,content_06dcc81394435cf5,True,True,True,True,116,1,587,...,0,0,0,0,0,0,1,2026-06,2025-10-02,2026-06-30
1152520,2026-06-29,client_b10cb2997d0c7c86,content_3822c9b7f03e95ec,True,True,True,False,2,0,14,...,0,0,0,0,0,0,0,2026-06,2026-02-10,2026-06-30
1152521,2026-06-29,client_b10cb2997d0c7c86,content_1fe4907594a9410d,True,True,True,False,13,0,360,...,0,0,0,0,0,0,0,2026-06,2026-03-03,2026-06-30
1152522,2026-06-29,client_b10cb2997d0c7c86,content_f78c8f862b9bb89d,True,True,True,False,7,0,42,...,0,0,0,0,0,0,0,2026-06,2026-03-03,2026-06-30


In [67]:
grouped = df.groupby([ 'client_hash_id','content_hash_id', 'month'])

numeric_cols = df.select_dtypes(include='number').columns.drop('gsc_avg_position')
bool_cols = df.select_dtypes(include='bool').columns

content_month = grouped.agg({
    **{col: 'sum' for col in numeric_cols},
    **{col: 'any' for col in bool_cols},
    'gsc_avg_position':'mean'
}).reset_index()

In [69]:
content_month

,client_hash_id,content_hash_id,month,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_avg_position
0,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-02,111,0,843,0,0,0,0,...,0,0,0,0,0,True,True,True,False,7.557559
1,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-03,330,1,2543,0,0,0,0,...,0,0,0,0,0,True,True,True,False,7.675690
2,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-04,393,5,2817,5,5,4,0,...,0,0,0,0,0,True,True,True,True,7.441862
3,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-05,872,1,6082,1,1,1,0,...,0,0,0,0,0,True,True,True,True,7.100545
4,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-06,1437,6,8742,8,8,8,1,...,0,0,0,0,1,True,True,True,True,6.115233
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91227,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-02,939,4,8060,5,5,5,0,...,0,0,0,0,0,True,True,True,True,8.860769
91228,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-03,1534,3,13546,6,6,6,0,...,0,0,0,0,0,True,True,True,True,9.573120
91229,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-04,482,0,9770,4,4,3,0,...,0,0,0,0,0,True,True,True,True,20.424837
91230,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-05,269,0,8335,5,5,5,1,...,0,0,0,0,1,True,True,True,True,32.566968


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.